# Retrieval: hybrid search + reranking

## Setup

Importing the retrieval stack and connecting to Qdrant, checking that the indexed collection exists and is populated before querying

In [ ]:
from qdrant_client import QdrantClient, models
from fastembed import SparseTextEmbedding
from sentence_transformers import SentenceTransformer, CrossEncoder

QDRANT_URL = "http://localhost:6333"
COLLECTION = "lecture_chunks"

DENSE_MODEL = "BAAI/bge-m3"
SPARSE_MODEL = "Qdrant/bm25"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"

client = QdrantClient(url=QDRANT_URL)
print("Collections:", [c.name for c in client.get_collections().collections])
print("Punkte in", COLLECTION, ":", client.count(COLLECTION).count)

Loading the dense (BGE-M3) and sparse (BM25) encoders, the same models used at indexing time, so query and document vectors stay comparable

In [ ]:
dense_embedder = SentenceTransformer(DENSE_MODEL, device="cuda")
sparse_embedder = SparseTextEmbedding(model_name=SPARSE_MODEL, language="german")
print("Dense + Sparse geladen.")

## Embed one query

Encoding one example query with both encoders and inspecting the vectors, just to make the dense and sparse query representations concrete

In [ ]:
query = "Wie funktioniert eine Faltung und warum ist das relevant?"

q_dense = dense_embedder.encode(query, normalize_embeddings=True)
q_sparse = list(sparse_embedder.query_embed(query))[0]

print("Dense-Vektor:", q_dense.shape, q_dense.dtype)
print("Sparse-Tokens:", len(q_sparse.indices), "Non-Zero")
print("Erste Token-IDs: ", q_sparse.indices[:8])
print("Erste BM25-Werte:", q_sparse.values[:8])

## Dense only and sparse only search

In [ ]:
def dense_only(query, limit=5):
    q = dense_embedder.encode(query, normalize_embeddings=True)
    return client.query_points(
        collection_name=COLLECTION, query=q.tolist(), using="dense",
        limit=limit, with_payload=True,
    ).points

for i, r in enumerate(dense_only(query), start=1):
    print(f"[{i}] {r.score:.4f} | {r.payload['lecture']} p.{r.payload['page_numbers']} | {r.payload['title']}")

In [ ]:
def sparse_only(query, limit=5):
    q = list(sparse_embedder.query_embed(query))[0]
    return client.query_points(
        collection_name=COLLECTION,
        query=models.SparseVector(indices=q.indices.tolist(), values=q.values.tolist()),
        using="sparse", limit=limit, with_payload=True,
    ).points

for i, r in enumerate(sparse_only(query), 1):
    print(f"[{i}] {r.score:.4f} | {r.payload['lecture']} p.{r.payload['page_numbers']} | {r.payload['title']}")

## Hybrid search (RRF)

hybrid_search runs the dense and sparse prefetches and fuses the two rankings with RRF

In [ ]:
def hybrid_search(query, limit=20, prefetch_limit=20):
    q_dense = dense_embedder.encode(query, normalize_embeddings=True)
    q_sparse = list(sparse_embedder.query_embed(query))[0]
    return client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            models.Prefetch(query=q_dense.tolist(), using="dense", limit=prefetch_limit),
            models.Prefetch(
                query=models.SparseVector(indices=q_sparse.indices.tolist(), values=q_sparse.values.tolist()),
                using="sparse", limit=prefetch_limit,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limit, with_payload=True,
    ).points

candidates = hybrid_search(query, limit=20)
print(f"{len(candidates)} Kandidaten. Top-10 nach RRF:\n")
for i, h in enumerate(candidates[:10], 1):
    print(f"[{i:2}] {h.score:.4f} | {h.payload['lecture']} p.{h.payload['page_numbers']} | {h.payload['title']}")

## Cross-encoder reranking

Loading the BGE cross-encoder reranker in fp16

In [ ]:
reranker = CrossEncoder(RERANK_MODEL, device="cuda")
reranker.model.half()  
print("Reranker geladen:", RERANK_MODEL, "| fp16")

## Reranking: top-20 to top-5

Scoring each [query, passage] pair with the cross-encoder and re-sorting the candidates, keeping the best 5 out of the 20 candidate pool

In [ ]:
def passage_text(payload):
    parts = [payload.get("title"), payload.get("page_content"), payload.get("context")]
    return "\n\n".join(p for p in parts if p)

pairs = [[query, passage_text(h.payload)] for h in candidates]
scores = reranker.predict(pairs, batch_size=16)

reranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)

print("Top-5 nach Reranking:\n")
for i, (h, s) in enumerate(reranked[:5], 1):
    print(f"[{i}] rerank={s:.4f} | {h.payload['lecture']} p.{h.payload['page_numbers']} | {h.payload['title']}")

## Before and after comparison

Printing a before/after table to show how strongly reranking reorders the hybrid results

In [ ]:
print(f"{'#':>2}  {'Hybrid (RRF)':<45}  {'→ nach Reranking':<45}")
print("-" * 96)
for i in range(5):
    h_hit = candidates[i]
    r_hit, r_score = reranked[i]
    h_str = f"{h_hit.payload['lecture']} p.{h_hit.payload['page_numbers']} {h_hit.payload['title']}"[:43]
    r_str = f"{r_hit.payload['lecture']} p.{r_hit.payload['page_numbers']} {r_hit.payload['title']}"[:43]
    moved = "" if h_hit.payload["chunk_id"] == r_hit.payload["chunk_id"] else "  <-- verschoben"
    print(f"{i+1:>2}  {h_str:<45}  {r_str:<45}{moved}")

## Full reusable retrieve() function

Wrapping hybrid retrieval plus reranking into a single retrieve() function, the exact interface that later moves into the backend

In [ ]:
def retrieve(query: str, top_k: int = 20, top_n: int = 5) -> list[dict]:
    candidates = hybrid_search(query, limit=top_k)
    if not candidates:
        return []
    pairs = [[query, passage_text(h.payload)] for h in candidates]
    scores = reranker.predict(pairs, batch_size=16)
    reranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)[:top_n]
    return [
        {
            "chunk_id": h.payload["chunk_id"],
            "lecture": h.payload["lecture"],
            "title": h.payload["title"],
            "page_numbers": h.payload["page_numbers"],
            "page_content": h.payload["page_content"],
            "context": h.payload.get("context"),
            "rerank_score": float(s),
        }
        for h, s in reranked
    ]

results = retrieve("Welche Kategorien unterscheidet der DBSCAN-Algorithmus?")
for i, r in enumerate(results, 1):
    print(f"[{i}] {r['rerank_score']:.4f} | {r['lecture']} p.{r['page_numbers']} | {r['title']}")